# Fresh run: Sprint 1 through Sprint 6

One Colab session, one checkout. Sprint 1 is the Overpass step that was failing (retired `kumi.systems` host, a shared User-Agent on private.coffee, the French whitelist, and the Swiss extract). This notebook stays on the branch that identifies the project and queries only global Overpass instances.

Earth Engine exports and U-Net training stay off until you set `INCLUDE_RASTER_EXPORT` and a GCP project id. GraphSAGE trains on the OSM graphs as soon as Sprint 1 writes them. Sprint 5 fuses those deprivation probabilities with a flood GeoTIFF when one is already on Drive; otherwise it stops and says which file is missing. Sprint 6 opens the dashboard from the policy table.


## 0. Checkout

Runtime → GPU is optional. GraphSAGE on the three settlements fits on CPU. The U-Net, when you turn it on later, is faster on a T4.


In [ ]:
%%bash
# One checkout for Sprints 1–6. This branch carries the Overpass client
# (project User-Agent; kumi, the French whitelist, and the Swiss extract
# are not queried). After that change is on main, set BRANCH=main.
BRANCH=cursor/overpass-identity-backoff-b69d
REPO=https://github.com/emmanuelamor5/nairobi-risk-mapping.git
cd /content
if [ ! -d nairobi-risk-mapping/.git ]; then
  git clone "$REPO" nairobi-risk-mapping
fi
cd nairobi-risk-mapping
git fetch origin "$BRANCH" || git fetch origin main
if git rev-parse --verify --quiet "origin/$BRANCH" >/dev/null; then
  git checkout "$BRANCH"
  git pull --ff-only origin "$BRANCH"
else
  echo "Branch $BRANCH is not on origin; using main."
  git checkout main
  git pull --ff-only origin main
fi
git rev-parse --abbrev-ref HEAD
git log -1 --oneline
grep -n OVERPASS_USER_AGENT src/config.py | head -3


In [ ]:
%cd /content/nairobi-risk-mapping
!pip install -q -r requirements.txt


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/nairobi_risk_mapping")
except ImportError:
    print("Not in Colab. Artefacts stay in the working directory.")
    DRIVE_ROOT = Path(".")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DATA_LINK = Path("data")
if not (DATA_LINK.is_symlink() or DATA_LINK.exists()):
    target = DRIVE_ROOT / "data"
    target.mkdir(parents=True, exist_ok=True)
    DATA_LINK.symlink_to(target)

# Raster export needs an Earth Engine project. Leave it False to run the
# OSM graph path only — that is the step the Overpass errors were blocking.
INCLUDE_RASTER_EXPORT = False
GEE_PROJECT_ID = None
GRAPHSAGE_EPOCHS = 100
SETTLEMENT_KEYS = ("kibera", "mathare", "mukuru")
print("DATA ->", DATA_LINK.resolve())


## 1. Sprint 1 — settlement graphs

Each settlement is fetched on its own. One mirror failure is recorded on that settlement and the next settlement still runs. The cell raises nothing that exits the runtime.


In [ ]:
from src import config
from src.data_pipeline import run_sprint1_all_settlements
from src.utils.graph_utils import configure_overpass_identity

print("Overpass User-Agent:", configure_overpass_identity())
print("Mirrors:", config.OVERPASS_URLS)

summary = run_sprint1_all_settlements(
    gee_project_id=GEE_PROJECT_ID,
    include_raster_export=INCLUDE_RASTER_EXPORT,
)
for key, result in summary.items():
    print(key, "->", result.get("status"))
    if result.get("status") == "ok":
        print("   graph:", result.get("graph_integrity"))
    else:
        print("   error:", result.get("error"))


## 2. Sprints 3 and 4 — GraphSAGE

Trains on every settlement graph Sprint 1 managed to write. Per-settlement softmax files are what Sprint 5 reads.


In [ ]:
from pathlib import Path

import torch

from src import config
from src.models.graphsage_model import GraphSAGEModel
from src.models.multi_settlement import (
    evaluate_graphsage_by_settlement,
    load_settlement_graphs,
    prepare_multi_settlement_data,
)

graphs = load_settlement_graphs(Path("data/graphs"), SETTLEMENT_KEYS)
print("Loaded:", {key: int(graph.num_nodes) for key, graph in graphs.items()} or "none")
if not graphs:
    raise RuntimeError(
        "No settlement graphs in data/graphs. The Sprint 1 Overpass cell has to succeed first."
    )

combined = prepare_multi_settlement_data(graphs, seed=42)
model = GraphSAGEModel(
    in_channels=int(combined.x.size(1)),
    hidden_channels=64,
    num_classes=config.DEPRIVATION_NUM_CLASSES,
    dropout=0.5,
)
checkpoint_dir = Path("checkpoints/sprint4/graphsage")
result = model.fit(
    combined,
    epochs=GRAPHSAGE_EPOCHS,
    learning_rate=5e-3,
    weight_decay=5e-4,
    checkpoint_dir=str(checkpoint_dir),
    early_stopping_patience=25,
    seed=42,
)
print("best_val_macro_f1:", result["best_val_macro_f1"])
print("sprint 3 bar:", result["meets_sprint3_target"], "target", result["sprint3_target"])
print("sprint 4 bar:", result["meets_final_target"], "target", result["final_target"])
if "val_by_settlement" in result:
    report = evaluate_graphsage_by_settlement(model, combined, split="val")
    for key, metrics in report["per_settlement"].items():
        print(f"  {key} val macro-F1 {metrics['macro_f1']:.3f}")

probs = model.predict_proba(combined).numpy()
settlement_ids = combined.settlement_id.detach().cpu().numpy()
for sid, key in enumerate(combined.settlement_keys):
    part = torch.tensor(probs[settlement_ids == sid])
    out = checkpoint_dir / f"{key}_deprivation_probs.pt"
    torch.save({"probs": part}, out)
    print("saved", out, tuple(part.shape))


## 3. Sprints 2 and 4 — flood head

The Attention U-Net needs the Earth Engine raster stack. This section only reports whether those tiles are on disk. Train them from `02_sprint2_attention_unet_v2.ipynb` and section 4 of `04_sprint4_three_settlements.ipynb`. Both notebooks now stay on this checkout.


In [ ]:
from pathlib import Path

for key in SETTLEMENT_KEYS:
    graph = Path("data/graphs") / f"{key}_graph.pt"
    tiles = Path("data/tiles") / key / "manifest_train.json"
    print(f"{key}: graph={graph.exists()}  train_tiles={tiles.exists()}")


## 4. Sprint 5 — fuse the heads

Deprivation probabilities come from the GraphSAGE cell. Flood probabilities come from a `{settlement}_flood_proba` GeoTIFF on Drive or under `data/raster_stacks`. Settlements without that raster are listed and left out of the table. No flood score is invented.


In [ ]:
from pathlib import Path

import numpy as np
import torch

from src import config
from src.models.fusion import (
    decide_policy,
    justice_audit,
    newest_geotiff,
    sample_geotiff_at_pos,
    save_policy_table,
)
from src.models.multi_settlement import load_settlement_graphs

search_dirs = [
    Path("data/raster_stacks"),
    DRIVE_ROOT,
    DRIVE_ROOT / "data" / "raster_stacks",
    DRIVE_ROOT / "checkpoints" / "sprint5",
]
graphs = load_settlement_graphs(Path("data/graphs"), SETTLEMENT_KEYS)
floods, deps, settlement_index, used = [], [], [], []
for sid, key in enumerate(graphs):
    graph = graphs[key]
    prob_path = Path("checkpoints/sprint4/graphsage") / f"{key}_deprivation_probs.pt"
    if graph.pos is None or not prob_path.exists():
        print(f"{key}: needs centroids and {prob_path.name}")
        continue
    try:
        blob = torch.load(prob_path, map_location="cpu", weights_only=False)
    except TypeError:
        blob = torch.load(prob_path, map_location="cpu")
    deprivation = blob["probs"].detach().cpu().numpy()
    raster = newest_geotiff(search_dirs, f"{key}_flood_proba")
    if raster is None:
        print(f"{key}: no {key}_flood_proba GeoTIFF. Run the U-Net notebooks, then rerun this cell.")
        continue
    flood = sample_geotiff_at_pos(
        raster,
        graph.pos.detach().cpu().numpy(),
        getattr(graph, "pos_crs", config.CRS_METRIC),
    )
    if flood.shape[0] != deprivation.shape[0]:
        print(f"{key}: flood length {flood.shape[0]} != deprivation {deprivation.shape[0]}")
        continue
    floods.append(flood)
    deps.append(deprivation)
    settlement_index.append(np.full(flood.shape[0], sid, dtype=np.int64))
    used.append(key)
    print(f"{key}: fusing {flood.shape[0]} parcels from {raster}")

if used and len(used) == len(graphs):
    decision = decide_policy(np.concatenate(floods), np.concatenate(deps, axis=0))
    audit = justice_audit(decision)
    table_path = save_policy_table(
        Path("data/policy_table.npz"),
        decision,
        settlement_index=np.concatenate(settlement_index),
        settlement_keys=used,
    )
    print("saved", table_path)
    print("justice audit passes:", audit["passes"], "violations:", audit["n_violations"])
else:
    print("Sprint 5 table not written.")
    print("Settlements with both heads:", used or "none")
    print("Notebook 05_sprint5_late_fusion.ipynb is the same checkout if you want the river-corridor lift.")


## 5. Sprint 6 — dashboard

The map is served only after Firebase sign-in. This cell builds a short preview and starts the server when `data/policy_table.npz` exists.


In [ ]:
from pathlib import Path

import numpy as np

from src.dashboard.app import _load_records, launch
from src.dashboard.map_view import build_policy_map
from src.models.fusion import resolve_policy_table

try:
    table_path = resolve_policy_table()
except FileNotFoundError as exc:
    print(exc)
    print("Sprint 6 waits on the Sprint 5 policy table.")
else:
    records = _load_records(str(table_path), "data/graphs")
    n_preview = min(5, int(records["positions"].shape[0]))
    preview = build_policy_map(
        records["positions"][:n_preview],
        records["action_index"][:n_preview],
        flood_proba=np.asarray(records["flood_proba"])[:n_preview],
        deprivation_class=np.asarray(records["deprivation_class"])[:n_preview],
        settlement_index=np.asarray(records["settlement_index"])[:n_preview],
        settlement_keys=[str(key) for key in records["settlement_keys"]],
        pos_crs=records["pos_crs"],
    )
    print("parcels", records["positions"].shape[0], "preview centre", preview.location)
    launch(records, port=8765)


## Done

Sprint 1 graphs and the GraphSAGE head are the part this session can finish without Earth Engine. The flood head, the fusion table, and the dashboard follow once a flood GeoTIFF is on Drive. Real accuracy still has to be read off that Colab run.
